## Constructed rolling out-of-sample allocation

Every newly computed value is a constructed method demonstration, not a market observation or a reproduction of historical results. French raw, extracted, and processed data are not bundled. The rights and permitted uses for French data have not been established, so this notebook does not analyze those data. Existing cell outputs, if present, are saved records and are not newly calculated by this walkthrough.


# Project 3: a constructed rolling comparison of four long-only portfolios

Seed 103 generates 48 months from 2000-01 for four independent assets A/B/C/D, with normal monthly simple returns of mean 0.005 and standard deviation 0.03; RF is fixed at 0.001. The notebook calls shared project functions with a 24-month window, Ledoit–Wolf covariance, and lag 0. The 0.4 cap applies only to minimum variance and maximum Sharpe; ERC is uncapped. Holdings cover 2002-01 through 2003-12: 24 months per strategy and 96 ledger rows if completed. It does not load the frozen historical configuration to run French research. The default writes six constructed files. The complete 440-month W60 historical report, exercises, and solutions are linked in the reader guide; this constructed W24 example is separate.


In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("project03_portfolio_oos_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
from IPython.display import display
sys.path.insert(0,str(ROOT / "projects/project03/src"))
from project03_backtest import STRATEGIES, Spec, run_strategy, apply_costs, metrics
from quant_course.performance import wealth_index
idx = pd.period_range("2000-01",periods=48,freq="M")
rng = np.random.default_rng(103)
returns = pd.DataFrame(rng.normal(0.005,0.03,(48,4)),index=idx,columns=list("ABCD"))
rf = pd.Series(0.001,index=idx)
spec = Spec(window=24,max_weight=0.4)
print("CONSTRUCTED OOS method; French branch SKIPPED: rights unknown")


## 1. Run four strategies under one constructed specification

Use a 24-month training window and a 10 bp weight-turnover fee proxy. This does not run the full historical parameter grid or promise a 25-second runtime.


In [ ]:
ledgers = {}; weights = {}
for strategy in STRATEGIES:
    ledger, weights[strategy] = run_strategy(strategy,returns,rf,spec,"2002-01","2003-12")
    ledgers[strategy] = apply_costs(ledger,10.0)
primary = pd.DataFrame({s:metrics(l) for s,l in ledgers.items()}).T
display(primary)
primary.to_csv(RUN_DIR / "constructed_primary_metrics.csv")


## 2. Skip the historical summary comparison

Constructed inputs cannot reproduce the French historical summary.json. The next cell only reports that it is skipped.


In [ ]:
print("Historical French summary comparison SKIPPED: this constructed panel cannot reproduce market results")


## 3. Ledger identities and timing

The first purchase starts from cash, giving a turnover proxy of 1; later trades follow weight drift. Assertions check net returns and the training cutoff. Four ledgers are written to the project RUN_DIR.


In [ ]:
led = ledgers["max_sharpe"]
assert np.allclose(led["net_return"], (1-led["cost"])*(1+led["gross_return"])-1)
assert (pd.PeriodIndex(led["train_end"],freq="M") < pd.PeriodIndex(led["holding_month"],freq="M")).all()
display(led[["holding_month","train_start","train_end","turnover","gross_return","cost","net_return"]].head())
for s,l in ledgers.items(): l.to_csv(RUN_DIR / (s+"_ledger_CONSTRUCTED.csv"),index=False)


## 4. Constructed net wealth

The series includes initial wealth of 1: 25 points for 24 holding months. The original Holding month axis shows positions 0 through 24, with 0 before the first holding month, rather than calendar dates. The vertical wealth scale is linear. The image is written to the project RUN_DIR.


In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
for s,l in ledgers.items(): ax.plot(wealth_index(l["net_return"].to_numpy()),label=s)
ax.set(title="CONSTRUCTED rolling allocation",xlabel="Holding month",ylabel="Net wealth"); ax.legend()
fig.tight_layout(); fig.savefig(RUN_DIR / "project03_wealth_CONSTRUCTED.png",dpi=130); plt.show()


## 5. Skip the historical block bootstrap and sensitivity grid

The next cell only prints a skip message; it computes no bootstrap intervals. See the complete historical report for the 5,000-resample, 12-month-block results.


In [ ]:
print("Full historical bootstrap/sensitivity grid SKIPPED; the small constructed path checks timing and cost accounting only")


## 6. Status counts for this constructed path

The last cell groups this constructed ledger by strategy and status. It does not estimate a hindsight portfolio. The historical hindsight comparison uses the entire evaluation period and is not tradable; it appears only in the report.


In [ ]:
print(pd.concat([l.assign(strategy=s) for s,l in ledgers.items()]).groupby(["strategy","status"]).size())
